<a href="https://colab.research.google.com/github/ishagt/ML_labs/blob/main/logisticregression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

In [ ]:
%pip install kagglehub[pandas-datasets]

In [ ]:
import kagglehub
import pandas as pd
import os
import tempfile
dataset_handle = "ivansabik/mexican-federal-government-salaries"
temp_dir = tempfile.mkdtemp()

print(f"Downloading dataset '{dataset_handle}' to temporary directory: {temp_dir}")
local_dataset_path = kagglehub.dataset_download(dataset_handle, output_dir=temp_dir)

print(f"Dataset downloaded to: {local_dataset_path}")
dataset_files = os.listdir(local_dataset_path)
print(f"Files in dataset: {dataset_files}")
df = None
for file_name in dataset_files:
    if file_name.endswith('.csv'):
        full_csv_path = os.path.join(local_dataset_path, file_name)
        df = pd.read_csv(full_csv_path)
        print(f"Loaded '{file_name}' into DataFrame 'df'.")
        break

if df is None:
    raise FileNotFoundError("No CSV file found in the downloaded dataset.")
print("\nFirst 5 records:")
display(df.head())

100%|██████████| 112M/112M [00:01<00:00, 89.5MB/s]

Extracting files...


Dataset downloaded to: /tmp/tmp93bsnz10
Files in dataset: ['salaries.csv', '.complete']
Loaded 'salaries.csv' into DataFrame 'df'.

First 5 records:


,entidadfederativa,sujetoobligado,nombre,denominacion,montoneto,cargo,area,montobruto,idInformacion,periodoreportainicio,periodoreportafin
0,Hidalgo,Jaltocán,Adolfo Hernandez Hernandez,Fontanero,4000.00,Fontanero,OBRAS PUBLICAS,4254.00,16311845,01/01/2018,30/06/2018
1,Ciudad de México,Secretaría de Salud,ARELY SAMANTA CLEOFAS VELASCO,"AUXILIAR DE ENFERMERIA ""A""",12177.86,"AUXILIAR DE ENFERMERIA ""A""",H.G. ENRIQUE CABRERA,16092.00,16480190,01/01/2018,31/03/2018
2,Ciudad de México,Secretaría de Seguridad Ciudadana,MELODY OLIMPIC GONZALEZ MONTES,POLICIA PRIMERO,11652.00,POLICIA PRIMERO,SUBSECRETARIA DE OPERACION POLICIAL,16030.00,17599078,01/01/2020,31/03/2020
3,Federación,Autoridad Educativa Federal en la Ciudad de Mé...,ANGEL ALLENDE PULIDO,APOYO Y ASISTENCIA A LA EDUCACION,10180.57,APOYO Y ASISTENCIA A LA EDUCACION,DIRECCIÓN GENERAL DE OPERACIONES DE SERVICIOS ...,2910.65,6514612,01/07/2018,31/12/2018
4,Aguascalientes,MUNICIPIO DE RINCÓN DE ROMOS,Yolanda Reyes Gonzalez,DIRECTOR,17004.40,DIRECTOR,ACCION CIVICA,6188.40,11927166,01/07/2019,31/12/2019


In [ ]:
# Display the first few rows of the DataFrame
print("First 5 records:")
display(df.head())
print("\nDataFrame Info:")
df.info()
print("\nDataFrame Description:")
display(df.describe())

First 5 records:


,entidadfederativa,sujetoobligado,nombre,denominacion,montoneto,cargo,area,montobruto,idInformacion,periodoreportainicio,periodoreportafin
0,Hidalgo,Jaltocán,Adolfo Hernandez Hernandez,Fontanero,4000.00,Fontanero,OBRAS PUBLICAS,4254.00,16311845,01/01/2018,30/06/2018
1,Ciudad de México,Secretaría de Salud,ARELY SAMANTA CLEOFAS VELASCO,"AUXILIAR DE ENFERMERIA ""A""",12177.86,"AUXILIAR DE ENFERMERIA ""A""",H.G. ENRIQUE CABRERA,16092.00,16480190,01/01/2018,31/03/2018
2,Ciudad de México,Secretaría de Seguridad Ciudadana,MELODY OLIMPIC GONZALEZ MONTES,POLICIA PRIMERO,11652.00,POLICIA PRIMERO,SUBSECRETARIA DE OPERACION POLICIAL,16030.00,17599078,01/01/2020,31/03/2020
3,Federación,Autoridad Educativa Federal en la Ciudad de Mé...,ANGEL ALLENDE PULIDO,APOYO Y ASISTENCIA A LA EDUCACION,10180.57,APOYO Y ASISTENCIA A LA EDUCACION,DIRECCIÓN GENERAL DE OPERACIONES DE SERVICIOS ...,2910.65,6514612,01/07/2018,31/12/2018
4,Aguascalientes,MUNICIPIO DE RINCÓN DE ROMOS,Yolanda Reyes Gonzalez,DIRECTOR,17004.40,DIRECTOR,ACCION CIVICA,6188.40,11927166,01/07/2019,31/12/2019



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1978155 entries, 0 to 1978154
Data columns (total 11 columns):
 #   Column                Dtype  
---  ------                -----  
 0   entidadfederativa     object 
 1   sujetoobligado        object 
 2   nombre                object 
 3   denominacion          object 
 4   montoneto             float64
 5   cargo                 object 
 6   area                  object 
 7   montobruto            float64
 8   idInformacion         int64  
 9   periodoreportainicio  object 
 10  periodoreportafin     object 
dtypes: float64(2), int64(1), object(8)
memory usage: 166.0+ MB

DataFrame Description:


,montoneto,montobruto,idInformacion
count,1.888979e+06,1.938733e+06,1.978155e+06
mean,1.353363e+04,1.677970e+04,1.564438e+07
std,1.446046e+05,2.443343e+04,1.185418e+07
min,-2.285230e+05,-2.586643e+05,6.600000e+01
25%,6.074895e+03,7.790960e+03,8.781188e+06
50%,9.802680e+03,1.224578e+04,1.425915e+07
75%,1.536840e+04,1.885615e+04,1.833432e+07
max,1.947336e+08,7.559543e+06,7.753209e+07


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
df_processed = df.copy()

print(f"Columns in df_processed before feature engineering: {df_processed.columns.tolist()}")
entry_date_col = 'fechadeingreso'

if entry_date_col in df_processed.columns:
    df_processed[entry_date_col] = pd.to_datetime(df_processed[entry_date_col], errors='coerce')
    latest_valid_date = df_processed[entry_date_col].max()
    if pd.isna(latest_valid_date):
        print(f"Warning: '{entry_date_col}' column contains no valid dates. Antiquity cannot be calculated.")
        df_processed['antiquity'] = np.nan # Assign NaN if no valid dates
    else:
        df_processed['antiquity'] = (latest_valid_date - df_processed[entry_date_col]).dt.days / 365.25
    print(f"'{entry_date_col}' processed. 'antiquity' column added.")
else:
    print(f"Warning: '{entry_date_col}' column not found. 'antiquity' cannot be calculated and will be NaN.")
    df_processed['antiquity'] = np.nan # Assign NaN if column not found

# --- 2. Target Creation: Define 'is_high_gross_income' ---
gross_income_col = 'ingresobruto'

if gross_income_col in df_processed.columns:
    # Convert 'ingresobruto' to numeric, coercing errors to NaN
    df_processed[gross_income_col] = pd.to_numeric(df_processed[gross_income_col], errors='coerce')

    # Calculate the median gross income
    median_gross_income = df_processed[gross_income_col].median()

    # Create a binary target: 1 if gross income is above median, 0 otherwise
    df_processed['is_high_gross_income'] = (df_processed[gross_income_col] > median_gross_income).astype(int)
    print(f"Target '{gross_income_col}' processed. 'is_high_gross_income' column added.")
else:
    raise ValueError(f"Target column '{gross_income_col}' not found in the dataset. Cannot create target variable.")

# --- 3. Select Features (X) and Target (y) ---
# Define numerical features that might be relevant
raw_feature_candidates = [
    'ingresonetos', # Net income
    'percepcionesextraordinarias', # Extraordinary perceptions
    'antiquity' # Years of service (created above)
]

target_column = 'is_high_gross_income'

# Filter out features that might not exist in df_processed
existing_feature_columns = [col for col in raw_feature_candidates if col in df_processed.columns]

if not existing_feature_columns:
    raise ValueError("No valid feature columns found after processing. Cannot proceed with model training.")

# Convert feature columns to numeric, coercing errors to NaN
for col in existing_feature_columns:
    df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce')

# Drop rows with any missing values in the selected features or target
columns_to_check = existing_feature_columns + [target_column]
initial_rows = len(df_processed)
df_processed.dropna(subset=columns_to_check, inplace=True)
dropped_rows = initial_rows - len(df_processed)
print(f"Dropped {dropped_rows} rows due to missing values in selected features/target.")

# Extract X and y
X = df_processed[existing_feature_columns].values
y = df_processed[target_column].values

# It's good practice to scale features for Logistic Regression
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Store selected_features for later use (e.g., plotting labels)
# This variable needs to be accessible globally after this cell runs
global selected_features
selected_features = existing_feature_columns

print(f"\nShape of X (features): {X.shape}")
print(f"Shape of y (target): {y.shape}")
print("\nFirst 5 rows of X (scaled features):")
display(X[:5])
print("\nFirst 5 values of y:")
display(y[:5])

Columns in df_processed before feature engineering: ['entidadfederativa', 'sujetoobligado', 'nombre', 'denominacion', 'montoneto', 'cargo', 'area', 'montobruto', 'idInformacion', 'periodoreportainicio', 'periodoreportafin']


ValueError: Target column 'ingresobruto' not found in the dataset. Cannot create target variable.

In [ ]:
for i in range(len(X)):
    print(X[i], y[i])

In [ ]:
model = LogisticRegression()
model.fit(X, y)

Option B: Using custom LogisticRegression()

In [ ]:
class CustomLogisticRegression:
    def __init__(self, learning_rate=0.01, num_iterations=10000):
        """
        Initializes the logistic regression model.
        """
        self.learning_rate = learning_rate
        self.num_iterations = num_iterations
        self.weights = None # This will store β1, β2, ..., βd
        self.bias = None    # This will store β0

    def _sigmoid(self, z):
        """
        The sigmoid activation function.
        Squashes continuous values into a 0 to 1 probability range.
        """
        # np.clip prevents overflow errors in np.exp() if z gets too largely negative
        z = np.clip(z, -250, 250)
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        """
        Trains the model using Gradient Descent to find the optimal separating hyperplane.
        """
        num_samples, num_features = X.shape

        # 1. Initialize weights (β1, β2) and bias (β0) to zero
        self.weights = np.zeros(num_features)
        self.bias = 0

        # 2. Gradient Descent Loop
        for _ in range(self.num_iterations):

            # Calculate linear combination: z = X*w + b
            linear_model = np.dot(X, self.weights) + self.bias

            # Apply sigmoid to get probabilities: ŷ = 1 / (1 + e^(-z))
            y_predicted = self._sigmoid(linear_model)
            dw = (1 / num_samples) * np.dot(X.T, (y_predicted - y))
            db = (1 / num_samples) * np.sum(y_predicted - y)
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

    def predict_proba(self, X):
        """
        Returns the probability of the positive class (Class 1).
        """
        linear_model = np.dot(X, self.weights) + self.bias
        return self._sigmoid(linear_model)

    def predict(self, X, threshold=0.5):
        """
        Predicts class labels based on a probability threshold (default 0.5).
        """
        probabilities = self.predict_proba(X)
        return np.array([1 if p >= threshold else 0 for p in probabilities])


In [ ]:
if __name__ == "__main__":
    clf = CustomLogisticRegression(learning_rate=0.01, num_iterations=5000)
    clf.fit(X, y)

    print("Custom Model Trained!")
    print(f"Intercept (β0): {clf.bias:.4f}")
    print("Coefficients (β): " + ", ".join([f"{w:.4f}" for w in clf.weights]))
    print("-" * 40)
    num_features_X = X.shape[1]

    if num_features_X == 3:
        new_samples_scaled = np.array([[ 0.5,  0.2,  1.0],  # Example 1: Slightly higher than average for all features
                                       [-1.0, -0.5, -0.8],  # Example 2: Lower than average
                                       [ 0.0,  0.0,  0.0]]) # Example 3: Average
    elif num_features_X == 2:
        new_samples_scaled = np.array([[ 0.5,  0.2],  # Example 1
                                       [-1.0, -0.5],  # Example 2
                                       [ 0.0,  0.0]]) # Example 3
    else:
        print(f"Warning: Cannot create sample predictions as X has {num_features_X} features. Expected 2 or 3.")
        new_samples_scaled = np.array([])

    if new_samples_scaled.size > 0:
        predictions = clf.predict(new_samples_scaled)
        probs = clf.predict_proba(new_samples_scaled)

        print("Predictions for new samples (scaled features):")
        for i, sample in enumerate(new_samples_scaled):
            print(f"Sample {sample} -> P(y=1): {probs[i]:.4f} | Predicted Class: {predictions[i]}")
    else:
        print("No sample predictions generated due to feature mismatch.")

In [ ]:
model = LogisticRegression()
model.fit(X, y)

beta_0 = model.intercept_[0]
coefficients = model.coef_[0]

print("Sklearn Model trained successfully!")
print(f"Intercept (β0): {beta_0:.4f}")
print("Coefficients (β): " + ", ".join([f"{c:.4f}" for c in coefficients]))

In [ ]:

num_features_X = X.shape[1]

if num_features_X == 3:
    new_sample_scaled = np.array([[0.8, 0.3, 0.5]]) # Example scaled values for 3 features
elif num_features_X == 2:
    new_sample_scaled = np.array([[0.8, 0.3]]) # Example scaled values for 2 features
else:
    print(f"Warning: Cannot create sample prediction as X has {num_features_X} features. Expected 2 or 3.")
    new_sample_scaled = np.array([])

if new_sample_scaled.size > 0:
    prediction = model.predict(new_sample_scaled)
    probability = model.predict_proba(new_sample_scaled)

    print(f"Prediction for scaled point {new_sample_scaled[0]}: Class {prediction[0]}")
    print(f"Probability [Class 0, Class 1]: {probability[0]}")
else:
    print("No sample prediction generated due to feature mismatch.")

In [ ]:
import matplotlib.pyplot as plt
if X.shape[1] >= 2:
    plt.figure(figsize=(10, 8))
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', marker='o', s=50, cmap=plt.cm.coolwarm, alpha=0.6)
    plt.title(f'Data Points from Mexican Government Salaries Dataset (First two features)')
    plt.xlabel(f'Feature 1: {selected_features[0]} (scaled)')
    plt.ylabel(f'Feature 2: {selected_features[1]} (scaled)')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.colorbar(label='Target: is_high_gross_income (0=low, 1=high)')
    plt.show()
else:
    print("Cannot plot decision boundary in 2D as X has less than two features.")